# Population Definition

**Objetivo**

Este notebook tem como objetivo:

- Entender a estrutura das bases disponibilizadas;
- Validar a qualidade e integridade dos dados;
- Analisar os relacionamentos entre as tabelas;
- Definir a população ativa para modelagem;
- Construir os datasets iniciais que serão utilizados nas próximas etapas do projeto.

## 1. Imports e Configurações

In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Ensure the repository root is on sys.path so `src` imports resolve
root = Path.cwd()
while not (root / "src").exists() and root.parent != root:
    root = root.parent

sys.path.insert(0, str(root))
print(f"Repository root added to sys.path: {root}")

# use project utilities
from src.data_loader import (
    load_base_cadastral,
    load_base_submissao,
    load_historico_emprestimos,
    load_historico_parcelas,
)
from src.population import build_active_population, build_population_score

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# load datasets via `src` helpers
base_cadastral = load_base_cadastral()
base_submissao = load_base_submissao()
historico_emprestimos = load_historico_emprestimos()
historico_parcelas = load_historico_parcelas()

OUTPUT_PATH = "../data/processed"

Repository root added to sys.path: g:\Meu Drive\Projetos\Github\cases\case_datarisk


## 2. Carregamento das Bases

### 2.1 Leitura dos Arquivos

In [2]:
base_cadastral.head()

,id_cliente,sexo,data_nascimento,qtd_filhos,qtd_membros_familia,renda_anual,tipo_renda,ocupacao,tipo_organizacao,nivel_educacao,estado_civil,tipo_moradia,possui_carro,possui_imovel,nota_regiao_cliente,nota_regiao_cliente_cidade
0,100023,F,1994-01-30,1,2.0,90000.0,State servant,Core staff,Kindergarten,Higher education,Single / not married,House / apartment,N,Y,2,2
1,100031,F,1973-11-13,0,1.0,112500.0,Working,Cooking staff,Business Entity Type 3,Secondary / secondary special,Widow,House / apartment,N,Y,3,2
2,100056,M,1975-02-19,0,2.0,360000.0,Working,Laborers,Transport: type 2,Secondary / secondary special,Married,House / apartment,Y,Y,2,2
3,100069,M,1986-04-10,1,2.0,360000.0,Working,Laborers,Transport: type 4,Secondary / secondary special,Separated,House / apartment,Y,Y,2,2
4,100085,M,1994-07-05,1,3.0,157500.0,Working,Drivers,Business Entity Type 1,Secondary / secondary special,Married,House / apartment,N,Y,2,2


In [3]:
base_submissao.head()

,id_cliente,data_solicitacao,dia_semana_solicitacao,hora_solicitacao,tipo_contrato,valor_credito,valor_bem,valor_parcela
0,100023,2025-02-24,MONDAY,12,Cash loans,544491.0,454500.0,17563.5
1,100031,2025-02-17,MONDAY,9,Cash loans,979992.0,702000.0,27076.5
2,100056,2025-02-20,THURSDAY,10,Cash loans,1506816.0,1350000.0,49927.5
3,100069,2025-02-10,MONDAY,11,Cash loans,640458.0,517500.0,27265.5
4,100085,2025-02-19,WEDNESDAY,12,Cash loans,755190.0,675000.0,28894.5


In [4]:
historico_emprestimos.head()

,id_contrato,id_cliente,tipo_contrato,status_contrato,data_decisao,data_liberacao,data_primeiro_vencimento,data_ultimo_vencimento_original,data_ultimo_vencimento,data_encerramento,valor_solicitado,valor_credito,valor_bem,valor_parcela,valor_entrada,percentual_entrada,qtd_parcelas_planejadas,taxa_juros_padrao,taxa_juros_promocional,tipo_pagamento,finalidade_emprestimo,tipo_cliente,faixa_rendimento,tipo_portfolio,tipo_produto,categoria_bem,combinacao_produto,setor_vendedor,canal_venda,area_venda,dia_semana_solicitacao,hora_solicitacao,flag_ultima_solicitacao_contrato,flag_ultima_solicitacao_dia,motivo_recusa,acompanhantes_cliente,flag_seguro_contratado
0,2802425,108129,Cash loans,Approved,2024-08-29,NaT,2024-09-28,2027-08-14,NaT,NaT,607500.0,679671.0,607500.0,25188.615,NaN,NaN,36.0,NaN,NaN,XNA,XNA,Repeater,low_action,Cash,x-sell,XNA,Cash X-Sell: low,XNA,Contact center,-1,THURSDAY,11,Y,1,XAP,Unaccompanied,1.0
1,2330894,258628,Cash loans,Approved,2022-10-11,NaT,2022-11-10,2024-09-30,2024-08-01,2024-08-04,148500.0,174361.5,148500.0,12165.210,NaN,NaN,24.0,NaN,NaN,Cash through the bank,XNA,Repeater,high,Cash,x-sell,XNA,Cash X-Sell: high,XNA,Credit and cash offices,-1,TUESDAY,15,Y,1,XAP,Unaccompanied,1.0
2,1182516,267782,Cash loans,Approved,2023-04-08,NaT,2023-05-08,2025-09-24,NaT,NaT,405000.0,451777.5,405000.0,20361.600,NaN,NaN,30.0,NaN,NaN,Cash through the bank,XNA,Repeater,low_normal,Cash,x-sell,XNA,Cash X-Sell: low,XNA,Credit and cash offices,-1,SATURDAY,4,Y,1,XAP,NaN,1.0
3,1543131,275707,Cash loans,Approved,2024-02-08,NaT,2024-03-09,2025-02-02,2025-02-02,2025-02-09,229500.0,241920.0,229500.0,22619.520,NaN,NaN,12.0,NaN,NaN,Cash through the bank,XNA,Repeater,low_normal,Cash,x-sell,XNA,Cash X-Sell: low,XNA,Credit and cash offices,-1,THURSDAY,8,Y,1,XAP,Unaccompanied,1.0
4,2261993,299391,Revolving loans,Canceled,2024-09-06,NaT,NaT,NaT,NaT,NaT,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,XNA,XAP,Repeater,XNA,XNA,XNA,XNA,Card Street,XNA,Credit and cash offices,-1,FRIDAY,13,Y,1,XAP,NaN,NaN


In [5]:
historico_parcelas.head()

,id_contrato,id_cliente,versao_parcela,numero_parcela,data_prevista_pagamento,data_real_pagamento,valor_previsto_parcela,valor_pago_parcela
0,1594684,100193,0.0,56,2021-12-21,2021-12-21,301.86,301.86
1,1995642,134723,1.0,38,2021-08-09,2021-08-04,12949.20,12949.20
2,1720935,176364,1.0,9,2024-03-06,2024-03-04,61192.53,61192.53
3,1439208,154898,1.0,20,2024-08-11,2024-08-07,8851.23,8851.23
4,1640082,172575,1.0,4,2022-11-15,2022-11-10,8720.28,8720.28


### 2.2 Visão Geral

In [6]:
bases = {
    "base_cadastral": base_cadastral,
    "base_submissao": base_submissao,
    "historico_emprestimos": historico_emprestimos,
    "historico_parcelas": historico_parcelas
}

overview = []

for nome, df in bases.items():

    overview.append({
        "base": nome,
        "linhas": df.shape[0],
        "colunas": df.shape[1]
    })

pd.DataFrame(overview)

,base,linhas,colunas
0,base_cadastral,40000,16
1,base_submissao,40000,8
2,historico_emprestimos,186890,37
3,historico_parcelas,1390978,8


**Principais Observações:**

- A base cadastral contém 40 mil clientes.
- A base de submissão contém 40 mil solicitações para score.
- O histórico possui mais de 186 mil contratos.
- O histórico de parcelas possui aproximadamente 1,4 milhão de registros.

O volume disponível é suficiente para construção de um modelo supervisionado robusto.

### 2.3 Missing Values

In [7]:
for nome, df in bases.items():

    print(f"\n{nome}")

    display(
        pd.DataFrame({
            "missing": df.isna().sum(),
            "missing_pct": (
                df.isna().mean() * 100
            ).round(2)
        })
        .query("missing > 0")
        .sort_values(
            "missing_pct",
            ascending=False
        )
    )


base_cadastral


,missing,missing_pct
ocupacao,12676,31.69



base_submissao


,missing,missing_pct
valor_bem,24,0.06
valor_parcela,4,0.01



historico_emprestimos


,missing,missing_pct
taxa_juros_padrao,186262,99.66
taxa_juros_promocional,186262,99.66
data_liberacao,179790,96.20
data_encerramento,100803,53.94
percentual_entrada,100168,53.60
valor_entrada,100168,53.60
data_ultimo_vencimento,99112,53.03
acompanhantes_cliente,92104,49.28
data_ultimo_vencimento_original,85747,45.88
data_primeiro_vencimento,79745,42.67



historico_parcelas


,missing,missing_pct
data_real_pagamento,339,0.02
valor_pago_parcela,339,0.02


**Principais Observações**

Foram identificadas variáveis com elevado percentual de ausência:

- taxa_juros_padrao
- taxa_juros_promocional
- data_liberacao

Essas variáveis serão reavaliadas nas etapas posteriores para definição de permanência ou exclusão do modelo.

## 3. Integridade Referencial

### 3.1 Clientes sem Cadastro

In [8]:
clientes_sem_cadastro = (
    set(historico_emprestimos["id_cliente"])
    -
    set(base_cadastral["id_cliente"])
)

print(
    f"Clientes sem cadastro: "
    f"{len(clientes_sem_cadastro)}"
)

Clientes sem cadastro: 0


### 3.2 Contratos sem Parcelas

In [9]:
contratos_sem_parcelas = (
    set(historico_emprestimos["id_contrato"])
    -
    set(historico_parcelas["id_contrato"])
)

print(
    f"Contratos sem parcelas: "
    f"{len(contratos_sem_parcelas)}"
)

Contratos sem parcelas: 79471


**Principais Observações**

- Todos os contratos possuem cliente cadastrado.
- Aproximadamente 79 mil contratos não possuem histórico de parcelas.

Esse comportamento sugere a existência de contratos recusados, cancelados ou não efetivados.

## 4. Tratamento de Datas

In [10]:
cols_emprestimos = [
    "data_decisao",
    "data_liberacao",
    "data_primeiro_vencimento",
    "data_ultimo_vencimento_original",
    "data_ultimo_vencimento",
    "data_encerramento"
]

for col in cols_emprestimos:
    historico_emprestimos[col] = pd.to_datetime(
        historico_emprestimos[col]
    )

for col in [
    "data_prevista_pagamento",
    "data_real_pagamento"
]:
    historico_parcelas[col] = pd.to_datetime(
        historico_parcelas[col]
    )

base_submissao["data_solicitacao"] = pd.to_datetime(
    base_submissao["data_solicitacao"]
)

base_cadastral["data_nascimento"] = pd.to_datetime(
    base_cadastral["data_nascimento"]
)

## 5. Cobertura Temporal

In [11]:
pd.DataFrame({
    "base": [
        "emprestimos",
        "parcelas"
    ],
    "data_min": [
        historico_emprestimos["data_decisao"].min(),
        historico_parcelas["data_prevista_pagamento"].min()
    ],
    "data_max": [
        historico_emprestimos["data_decisao"].max(),
        historico_parcelas["data_prevista_pagamento"].max()
    ]
})

,base,data_min,data_max
0,emprestimos,2017-02-04,2025-02-22
1,parcelas,2017-02-28,2025-02-22


**Principais Observações**

O histórico disponível cobre aproximadamente oito anos de operação (2017 a 2025), permitindo a construção de variáveis comportamentais e análise longitudinal dos clientes.

## 6. Análise da População

### 6.1 Status dos Contratos

In [12]:
(
    historico_emprestimos["status_contrato"]
    .value_counts()
    .to_frame()
)

,count
status_contrato,
Approved,116182
Canceled,35767
Refused,32108
Unused offer,2833


### 6.2 Contratos por Cliente

In [13]:
(
    historico_emprestimos
    .groupby("id_cliente")
    ["id_contrato"]
    .nunique()
    .describe()
)

count    37952.000000
mean         4.924378
std          4.189973
min          1.000000
25%          2.000000
50%          4.000000
75%          7.000000
max         66.000000
Name: id_contrato, dtype: float64

**Principais Observações**

A maioria dos contratos encontra-se na situação Approved.

Como o objetivo do projeto é modelar inadimplência, apenas contratos efetivamente concedidos serão considerados elegíveis para construção da variável target.

## 7. Definição da População Ativa

A população ativa é construída em nível de cliente + safra mensal (`id_cliente` + `safra_mes`). Portanto, um mesmo cliente pode aparecer em mais de uma linha quando solicitar ou receber crédito em meses diferentes.

A chave da população, porém, deve ser única dentro de cada safra. Se houver mais de um contrato aprovado para o mesmo `id_cliente` na mesma `safra_mes`, mantemos apenas o caso mais recente pela `data_decisao`; em caso de empate, usamos o maior `id_contrato` como desempate determinístico.

A mesma regra de unidade é aplicada à população de score. A `safra_mes` é derivada da `data_solicitacao` e, se um cliente possuir mais de uma solicitação no mesmo mês, mantemos a solicitação mais recente.

A base de submissão não é cruzada diretamente com o histórico de empréstimos; a população ativa é derivada da base cadastral e dos contratos aprovados do histórico.

A seleção considera:
- contratos aprovados (`status_contrato == 'Approved'`);
- contratos que possuem histórico de parcelas;
- contratos dentro do período de modelagem: janeiro de 2020 a janeiro de 2025;
- deduplicação por `id_cliente` + `safra_mes`, preservando o evento mais recente.

Além disso, construímos métricas históricas por linha de população, usando o histórico de empréstimos até a data de decisão da linha ativa ou a data de solicitação na população de score. Essas métricas respeitam a data de referência de cada linha, permitindo que o histórico de um cliente evolua entre safras diferentes. Elas incluem:
- `qtd_contratos_aceitos_historico`;
- `qtd_contratos_recusados_historico`;
- `soma_valor_credito_ativo_historico`;
- `media_valor_credito_aceito_historico`.

Se o cliente não possui histórico anterior, essas métricas são preenchidas com zero para evitar valores ausentes.

In [14]:
population_active = build_active_population()
population_score = build_population_score()

print(
    f"População ativa: {population_active.shape}"
)

print(
    f"Score: {population_score.shape}"
)

print(
    f"Clientes ativos selecionados: {population_active['id_cliente'].nunique()}"
)
print(
    f"Clientes de score mantidos: {population_score['id_cliente'].nunique()}"
)

População ativa: (81882, 28)
Score: (40000, 28)
Clientes ativos selecionados: 35353
Clientes de score mantidos: 40000


In [15]:
# Exibir as novas métricas históricas para as populações ativa e de score
population_active[
    [
        'id_cliente',
        'safra_mes',
        'qtd_contratos_aceitos_historico',
        'qtd_contratos_recusados_historico',
        'soma_valor_credito_ativo_historico',
        'media_valor_credito_aceito_historico',
    ]
].head()

population_score[
    [
        'id_cliente',
        'safra_mes',
        'qtd_contratos_aceitos_historico',
        'qtd_contratos_recusados_historico',
        'soma_valor_credito_ativo_historico',
        'media_valor_credito_aceito_historico',
    ]
].head()


,id_cliente,safra_mes,qtd_contratos_aceitos_historico,qtd_contratos_recusados_historico,soma_valor_credito_ativo_historico,media_valor_credito_aceito_historico
0,100023,2025-02,4,0,454068.0,113517.000000
1,100031,2025-02,0,0,0.0,0.000000
2,100056,2025-02,3,0,849991.5,283330.500000
3,100067,2025-02,7,16,367587.0,52512.428571
4,100069,2025-02,6,1,971617.5,161936.250000


## 8. Distribuição das Safras

`safra_mes` da população ativa é derivado de `data_decisao` na base de população ativa e `data_solicitacao` na base de submissão (base de score).

In [16]:
def build_vintage_distribution(df):
    distribution = (
        df.groupby("safra_mes")["id_cliente"]
        .nunique()
        .reset_index(name="contratos")
        .sort_values("safra_mes")
        .reset_index(drop=True)
    )
    total = distribution["contratos"].sum()
    distribution["percentual"] = (
        distribution["contratos"] / total * 100
    ).map(lambda value: f"{value:.2f}%")
    total_row = pd.DataFrame(
        {
            "safra_mes": ["Total"],
            "contratos": [total],
            "percentual": ["100.00%"],
        }
    )
    return pd.concat([distribution, total_row], ignore_index=True)


# Distribuição por safra mensal para avaliação da vigência da população ativa
build_vintage_distribution(population_active)

,safra_mes,contratos,percentual
0,2020-01,630,0.77%
1,2020-02,672,0.82%
2,2020-03,707,0.86%
3,2020-04,729,0.89%
4,2020-05,739,0.90%
5,2020-06,737,0.90%
6,2020-07,672,0.82%
7,2020-08,668,0.82%
8,2020-09,773,0.94%
9,2020-10,864,1.06%


In [17]:
build_vintage_distribution(population_score)

,safra_mes,contratos,percentual
0,2025-02,40000,100.00%
1,Total,40000,100.00%


In [18]:
population_active.head()

,id_cliente,data_solicitacao,dia_semana_solicitacao_submissao,hora_solicitacao_submissao,tipo_contrato_submissao,valor_credito_submissao,valor_bem_submissao,valor_parcela_submissao,sexo,data_nascimento,qtd_filhos,qtd_membros_familia,renda_anual,tipo_renda,ocupacao,tipo_organizacao,nivel_educacao,estado_civil,tipo_moradia,possui_carro,possui_imovel,nota_regiao_cliente,nota_regiao_cliente_cidade,safra_mes,qtd_contratos_aceitos_historico,qtd_contratos_recusados_historico,soma_valor_credito_ativo_historico,media_valor_credito_aceito_historico
0,100023,2020-02-01,SATURDAY,10,Consumer loans,93145.5,91282.5,7992.00,F,1994-01-30,1,2.0,90000.0,State servant,Core staff,Kindergarten,Higher education,Single / not married,House / apartment,N,Y,2,2,2020-02,2,0,169825.5,84912.75
1,100023,2024-06-07,FRIDAY,9,Cash loans,239242.5,180000.0,16822.44,F,1994-01-30,1,2.0,90000.0,State servant,Core staff,Kindergarten,Higher education,Single / not married,House / apartment,N,Y,2,2,2024-06,4,0,454068.0,113517.00
2,100067,2022-05-04,WEDNESDAY,11,Cash loans,90000.0,90000.0,9222.30,F,1996-09-03,1,3.0,162000.0,Working,Sales staff,Trade: type 2,Higher education,Civil marriage,House / apartment,Y,Y,2,2,2022-05,4,0,201231.0,50307.75
3,100067,2023-02-25,SATURDAY,12,Consumer loans,47101.5,52339.5,4780.35,F,1996-09-03,1,3.0,162000.0,Working,Sales staff,Trade: type 2,Higher education,Civil marriage,House / apartment,Y,Y,2,2,2023-02,5,1,248332.5,49666.50
4,100067,2023-07-06,THURSDAY,13,Consumer loans,50908.5,63639.0,5166.72,F,1996-09-03,1,3.0,162000.0,Working,Sales staff,Trade: type 2,Higher education,Civil marriage,House / apartment,Y,Y,2,2,2023-07,6,2,299241.0,49873.50


In [19]:
population_score.head()

,id_cliente,data_solicitacao,dia_semana_solicitacao_submissao,hora_solicitacao_submissao,tipo_contrato_submissao,valor_credito_submissao,valor_bem_submissao,valor_parcela_submissao,sexo,data_nascimento,qtd_filhos,qtd_membros_familia,renda_anual,tipo_renda,ocupacao,tipo_organizacao,nivel_educacao,estado_civil,tipo_moradia,possui_carro,possui_imovel,nota_regiao_cliente,nota_regiao_cliente_cidade,safra_mes,qtd_contratos_aceitos_historico,qtd_contratos_recusados_historico,soma_valor_credito_ativo_historico,media_valor_credito_aceito_historico
0,100023,2025-02-24,MONDAY,12,Cash loans,544491.0,454500.0,17563.5,F,1994-01-30,1,2.0,90000.0,State servant,Core staff,Kindergarten,Higher education,Single / not married,House / apartment,N,Y,2,2,2025-02,4,0,454068.0,113517.000000
1,100031,2025-02-17,MONDAY,9,Cash loans,979992.0,702000.0,27076.5,F,1973-11-13,0,1.0,112500.0,Working,Cooking staff,Business Entity Type 3,Secondary / secondary special,Widow,House / apartment,N,Y,3,2,2025-02,0,0,0.0,0.000000
2,100056,2025-02-20,THURSDAY,10,Cash loans,1506816.0,1350000.0,49927.5,M,1975-02-19,0,2.0,360000.0,Working,Laborers,Transport: type 2,Secondary / secondary special,Married,House / apartment,Y,Y,2,2,2025-02,3,0,849991.5,283330.500000
3,100067,2025-02-18,TUESDAY,14,Cash loans,45000.0,45000.0,5337.0,F,1996-09-03,1,3.0,162000.0,Working,Sales staff,Trade: type 2,Higher education,Civil marriage,House / apartment,Y,Y,2,2,2025-02,7,16,367587.0,52512.428571
4,100069,2025-02-10,MONDAY,11,Cash loans,640458.0,517500.0,27265.5,M,1986-04-10,1,2.0,360000.0,Working,Laborers,Transport: type 4,Secondary / secondary special,Separated,House / apartment,Y,Y,2,2,2025-02,6,1,971617.5,161936.250000


**Decisão Adotada**

A população ativa agora é definida por cliente + safra mensal, preservando o vínculo com o evento mais recente dentro de cada combinação `id_cliente` + `safra_mes`.
Isso permite que o mesmo cliente apareça em safras diferentes, mas impede duplicidade dentro do mesmo mês de referência.

- `population_active`: clientes elegíveis com contrato aprovado e parcela registrada, deduplicados por `id_cliente` + `safra_mes`.
- `population_score`: clientes de submissão com dados cadastrais a serem escorados.

A finalização dos cortes de treino, teste e OOT segue sendo feita após a construção da feature store.


## 9. Persistência

In [20]:
Path(OUTPUT_PATH).mkdir(
    parents=True,
    exist_ok=True
)

population_active.to_parquet(
    f"{OUTPUT_PATH}/population_active.parquet",
    index=False
)

population_score.to_parquet(
    f"{OUTPUT_PATH}/population_score.parquet",
    index=False
)

## 10. Conclusões

Esta etapa consolidou a definição das populações que serão usadas nas próximas fases do projeto, garantindo que a base de modelagem esteja coerente com o problema de negócio e com a forma como o crédito é solicitado ao longo do tempo.

**Principais achados:**

- A base cadastral possui 40 mil clientes e não apresentou problemas de integridade com o histórico de contratos.
- A base de submissão possui 40 mil solicitações para score, representando a população futura a ser escorada.
- O histórico de empréstimos contém mais de 186 mil contratos, e o histórico de parcelas contém aproximadamente 1,4 milhão de registros.
- O histórico cobre o período de 2017 a 2025, mas a população ativa foi restringida aos janeiro de 2020 a janeiro de 2025 para manter uma janela de modelagem mais estável e comparável.
- Cerca de 79 mil contratos não possuem histórico de parcelas, o que reforça a decisão de usar apenas contratos aprovados e efetivamente acompanhados por parcelas na população ativa.

**Definição adotada para a população:**

- A unidade final da população ativa é `id_cliente` + `safra_mes`, e não apenas `id_cliente`.
- Isso permite que um mesmo cliente apareça em safras diferentes quando tiver novos pedidos ou novos créditos em meses distintos.
- Dentro de uma mesma safra, a chave deve ser única; quando há mais de um evento para o mesmo cliente no mesmo mês, é mantido o caso mais recente.
- A população ativa considera contratos `Approved`, com histórico de parcelas e dentro da janela de 2020 a janeiro de 2025.
- A população de score segue a mesma lógica de safra, usando `data_solicitacao` para construir `safra_mes`.

**Variáveis históricas construídas:**

Foram adicionadas métricas históricas calculadas até a data de referência de cada linha da população. Isso é importante porque o histórico disponível para um cliente em 2020 não deve ser o mesmo histórico disponível para esse mesmo cliente em 2025.

As variáveis criadas foram:

- `qtd_contratos_aceitos_historico`;
- `qtd_contratos_recusados_historico`;
- `soma_valor_credito_ativo_historico`;
- `media_valor_credito_aceito_historico`.

Essas features tornam a população mais adequada para modelagem temporal, pois representam o conhecimento acumulado sobre o cliente até aquela safra específica.

**Resultado da etapa:**

- `population_active`: população elegível para desenvolvimento do modelo e definição posterior do target, em nível cliente + safra.
- `population_score`: população a ser escorada, com estrutura compatível com a população ativa.
- Ambas as populações seguem o mesmo esquema final de colunas, facilitando a aplicação consistente das regras de preparação, feature engineering e score.

A próxima etapa consiste em construir o target de inadimplência no mesmo nível da população ativa (`id_cliente` + `safra_mes`), garantindo que a variável resposta esteja alinhada à unidade de observação definida neste notebook.